# Notebook 03: ESM-2 Embedding Extraction

## HIV Drug Resistance Prediction with ESM-2

---

**Objective**: Extract protein language model embeddings from ESM-2.

**Model**: esm2_t33_650M_UR50D (650M parameters, 1280-dim embeddings)

**Approach**:
- Extract token-level embeddings from final layer
- Compare pooling strategies (mean, max, attention-weighted)

**Hardware**: GPU with 16+ GB VRAM recommended

---

In [ ]:
# Install ESM-2 if needed
# !pip install -q fair-esm torch

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from data_processing import load_fasta, load_unified_data
from feature_engineering import (
    load_esm2_model, extract_embeddings,
    batch_extract_pooled_embeddings,
    mean_pooling, max_pooling, mean_max_pooling
)

# Random seed
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Project paths
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
EMBEDDINGS_DIR = PROJECT_ROOT / 'data' / 'embeddings'

EMBEDDINGS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Embeddings will be saved to: {EMBEDDINGS_DIR}")

## 1. Load Data

In [ ]:
# Load unified data
unified_data = load_unified_data(DATA_DIR)

print("\nLoaded data:")
for dc, data in unified_data.items():
    n_seqs = len(data['sequences'])
    seq_lengths = [len(s) for s in data['sequences']]
    print(f"  {dc}: {n_seqs} sequences ({np.mean(seq_lengths):.0f} aa avg)")

## 2. Load ESM-2 Model

In [ ]:
print("Loading ESM-2 model...")

model, alphabet, batch_converter, device = load_esm2_model(
    model_name="esm2_t33_650M_UR50D"
)

print(f"\nModel loaded on: {device}")
print(f"Number of layers: {model.num_layers}")
print(f"Embedding dimension: 1280")

## 3. Extract Embeddings

Extract embeddings with different pooling strategies:
1. Mean pooling (average across positions)
2. Max pooling (maximum across positions)
3. Mean + Max concatenation

In [ ]:
# Configuration
BATCH_SIZE = 4  # Reduce if running out of GPU memory
POOLING_METHODS = ['mean', 'max', 'mean_max']

In [ ]:
# Extract embeddings for each drug class
for drug_class, data in unified_data.items():
    print(f"\n{'='*60}")
    print(f"EXTRACTING {drug_class} EMBEDDINGS")
    print(f"{'='*60}")
    
    sequences = data['sequences']
    n_seqs = len(sequences)
    
    for pooling_method in POOLING_METHODS:
        output_path = EMBEDDINGS_DIR / f'{drug_class}_pooled_{pooling_method}.npy'
        
        # Check if already exists
        if output_path.exists():
            print(f"\n{pooling_method}: Already exists, loading...")
            embeddings = np.load(output_path)
            print(f"  Shape: {embeddings.shape}")
            continue
        
        print(f"\n{pooling_method}: Extracting...")
        
        embeddings = batch_extract_pooled_embeddings(
            sequences,
            model, alphabet, batch_converter, device,
            pooling_method=pooling_method,
            batch_size=BATCH_SIZE,
            repr_layer=33
        )
        
        # Save embeddings
        np.save(output_path, embeddings)
        size_mb = embeddings.nbytes / 1e6
        print(f"  Shape: {embeddings.shape}")
        print(f"  Saved to: {output_path.name} ({size_mb:.1f} MB)")

## 4. Verify Embeddings

In [ ]:
print("\nVerifying saved embeddings:")
print("-" * 50)

for drug_class in unified_data.keys():
    print(f"\n{drug_class}:")
    
    for pooling_method in POOLING_METHODS:
        filepath = EMBEDDINGS_DIR / f'{drug_class}_pooled_{pooling_method}.npy'
        
        if filepath.exists():
            embeddings = np.load(filepath)
            size_mb = filepath.stat().st_size / 1e6
            
            # Check for NaN/Inf
            has_nan = np.isnan(embeddings).any()
            has_inf = np.isinf(embeddings).any()
            
            status = "OK" if not (has_nan or has_inf) else "ISSUE"
            print(f"  {pooling_method}: {embeddings.shape} - {size_mb:.1f} MB - {status}")
        else:
            print(f"  {pooling_method}: NOT FOUND")

## 5. Embedding Statistics

In [ ]:
import matplotlib.pyplot as plt

# Load mean pooled embeddings and visualize
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, drug_class in enumerate(unified_data.keys()):
    ax = axes[idx]
    
    embeddings = np.load(EMBEDDINGS_DIR / f'{drug_class}_pooled_mean.npy')
    
    # Plot embedding norm distribution
    norms = np.linalg.norm(embeddings, axis=1)
    ax.hist(norms, bins=50, edgecolor='white', alpha=0.7)
    ax.set_xlabel('Embedding L2 Norm')
    ax.set_ylabel('Count')
    ax.set_title(f'{drug_class}\nMean: {norms.mean():.2f} +/- {norms.std():.2f}')

plt.tight_layout()
plt.savefig(EMBEDDINGS_DIR / 'embedding_norms.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Dimensionality reduction visualization (PCA)
from sklearn.decomposition import PCA

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, drug_class in enumerate(unified_data.keys()):
    ax = axes[idx]
    
    embeddings = np.load(EMBEDDINGS_DIR / f'{drug_class}_pooled_mean.npy')
    
    # PCA
    pca = PCA(n_components=2)
    pca_coords = pca.fit_transform(embeddings)
    
    ax.scatter(pca_coords[:, 0], pca_coords[:, 1], alpha=0.5, s=10)
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
    ax.set_title(f'{drug_class} Embeddings (PCA)')

plt.tight_layout()
plt.savefig(EMBEDDINGS_DIR / 'embedding_pca.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

**Embeddings extracted:**
- Dimension: 1280 (mean/max) or 2560 (mean_max)
- Pooling methods: mean, max, mean_max
- Stored as .npy files in data/embeddings/

**Total size estimate:**
- ~6,308 sequences x 1280 dims x 4 bytes = ~32 MB per pooling method
- ~100 MB total for all pooled embeddings

**Next steps:**
- Notebook 04: Train classifiers and compare to baseline